In [1]:
!pip install datasets==3.6.0
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.19
    Uninstalling multiprocess-0.70.19:
      Successfully uninstalled multiprocess-0.70.19
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
ERROR: pip's dependency re

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import torch
import transformers
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModel, BertTokenizer, BertConfig, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from pathlib import Path
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from transformers import TrainerCallback

from transformers.activations import ACT2FN
from transformers.cache_utils import Cache
from transformers.modeling_layers import GradientCheckpointingLayer
from transformers.modeling_outputs import BaseModelOutputWithPastAndCrossAttentions
from transformers.processing_utils import Unpack
from transformers.pytorch_utils import apply_chunking_to_forward
from transformers.utils import TransformersKwargs, auto_docstring
from transformers.models.bert.modeling_bert import BertAttention, BertModel, BertSelfAttention, BertLayer, BertPreTrainedModel, BertForSequenceClassification

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available else "cpu"

config = BertConfig(
    vocab_size=130000,
    num_hidden_layers=8,
    hidden_size=512,
    num_attention_heads=8,
    max_position_embeddings=1024,
    num_labels=2,
)

dataset = load_dataset("RussianNLP/russian_super_glue", "russe", trust_remote_code=True)
df = dataset['train'].to_pandas()
train_df = df.iloc[:-1000].reset_index(drop=True)
val_df   = df.iloc[-1000:].reset_index(drop=True)
train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)

train_dataloader = DataLoader(train_dataset, 50, 1)
val_dataloader = DataLoader(val_dataset, 50, 1)
print(train_dataset)

loss_fn = nn.CrossEntropyLoss()

README.md:   0%|          | 0.00/35.8k [00:00<?, ?B/s]

russian_super_glue.py:   0%|          | 0.00/23.6k [00:00<?, ?B/s]

data/RUSSE.zip: reconstructing file:   0%|          |  0.00B / 3.81MB            

data/RUSSE.zip: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/19845 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/8505 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/18892 [00:00<?, ? examples/s]

Dataset({
    features: ['word', 'sentence1', 'sentence2', 'start1', 'start2', 'end1', 'end2', 'gold_sense1', 'gold_sense2', 'idx', 'label'],
    num_rows: 18845
})


In [3]:
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
import os

def angular_distance(x_l, x_ln):
    if isinstance(x_l, torch.Tensor):
        x_l = x_l.detach().cpu().numpy()
    if isinstance(x_ln, torch.Tensor):
        x_ln = x_ln.detach().cpu().numpy()

    x_l_flat = x_l.reshape(-1)
    x_ln_flat = x_ln.reshape(-1)

    dot = np.dot(x_l_flat, x_ln_flat)
    norm_l = np.linalg.norm(x_l_flat)
    norm_ln = np.linalg.norm(x_ln_flat)

    if norm_l == 0 or norm_ln == 0:
        return 1.0

    cos_sim = np.clip(dot / (norm_l * norm_ln), -1.0, 1.0)
    angle = np.arccos(cos_sim) / np.pi
    return angle


def layer_distance_matrix(hidden_states):
    n_layers = len(hidden_states)
    dist_matrix = np.zeros((n_layers, n_layers))

    for ii in range(n_layers):
        for jj in range(ii + 1, n_layers):
            dist = angular_distance(hidden_states[ii], hidden_states[jj])
            dist_matrix[ii, jj] = dist
            dist_matrix[jj, ii] = dist
        dist_matrix[ii, ii] = 0.0

    return dist_matrix

def get_all_layer_hidden_states(model, sentences1, sentences2, tokenizer):
    model.eval()

    encodings = tokenizer(
        sentences1,
        sentences2,
        padding=True,
        truncation=True,
        max_length=500,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.bert(
            **encodings,
            output_hidden_states=True,
            return_dict=True
        )
        hidden_states = outputs.hidden_states

    return list(hidden_states)

def plot_hidden_states_norms(layer_norms, file_name):
    save_dir = f"/content/drive/MyDrive/colab_results/{file_name}"
    os.makedirs(save_dir, exist_ok=True)
    layers = list(layer_norms.keys())
    norms = list(layer_norms.values())

    plt.figure(figsize=(12, 6))
    bars = plt.bar(layers, norms, color='lightsalmon', edgecolor='darkred', alpha=0.8)

    for bar in bars:
        height = int(bar.get_height())
        plt.text(bar.get_x() + bar.get_width()/2., height * 1.01,
                 f'{height}', ha='center', va='bottom', fontsize=9)

    plt.xlabel("Слой энкодера", fontsize=12)
    plt.ylabel("Норма выходных активаций", fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/output_hist", dpi=300, bbox_inches='tight')
    plt.close()

def ang_matr_processing(all_layer_states, file_name, font):
    save_dir = f"/content/drive/MyDrive/colab_results/{file_name}"
    os.makedirs(save_dir, exist_ok=True)
    dist_matrix = layer_distance_matrix(all_layer_states)

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        dist_matrix,
        annot=True,
        fmt=".3f",
        cmap="viridis",
        square=True,
        xticklabels=range(dist_matrix.shape[0]),
        yticklabels=range(dist_matrix.shape[0]),
        cbar_kws={"shrink": 0.8, "label": "Угловое расстояние"},
        annot_kws={"fontsize": font}
    )
    plt.xlabel("Индекс слоя")
    plt.ylabel("Индекс слоя")
    plt.tight_layout()
    plt.savefig(f"{save_dir}/ang_matr", dpi=300, bbox_inches='tight')
    plt.close()

    diag = np.diag(dist_matrix, k=1)
    mean = np.mean(diag)
    with open(f"{save_dir}/ang_mean.txt", 'w', encoding='utf-8') as f:
        f.write(str(mean))

def ang_hist_processing(all_layer_states, file_name):
    layer_norms = {}
    for layer_num, hs in enumerate(all_layer_states):
        norm = torch.norm(hs.reshape(-1), p=2).item()
        layer_key = str(layer_num)
        layer_norms[layer_key] = norm

    plot_hidden_states_norms(layer_norms, file_name)

    norms = list(layer_norms.values())
    output_var = np.var(norms)
    save_dir = f"/content/drive/MyDrive/colab_results/{file_name}"
    os.makedirs(save_dir, exist_ok=True)
    with open(f"{save_dir}/output_var.txt", 'w', encoding='utf-8') as f:
        f.write(str(output_var))

    print(f"Дисперсия норм выходных активаций: {output_var:.4f}")

In [5]:
def compute_layer_gradient_norms(model, loss):
    if not any(p.grad is not None for p in model.parameters()):
        loss.backward(retain_graph=True)

    layer_norms = {}
    layer_grads = defaultdict(list)

    for name, param in model.named_parameters():
        if param.grad is None:
            continue

        if 'bert.encoder.layer' in name:
            layer_num = name.split('layer.')[1].split('.')[0]
            layer_key = str(layer_num)
        else:
            continue

        layer_grads[layer_key].append(param.grad.reshape(-1))

    for layer_key, grads in layer_grads.items():
        all_grad = torch.cat(grads)
        norm = torch.norm(all_grad, p=2).item()
        layer_norms[layer_key] = norm

    return layer_norms

def plot_gradient_norms(layer_norms, file_name):
    save_dir = f"/content/drive/MyDrive/colab_results/{file_name}"
    os.makedirs(save_dir, exist_ok=True)
    layers = list(layer_norms.keys())
    norms = list(layer_norms.values())

    plt.figure(figsize=(12, 6))
    bars = plt.bar(layers, norms, color='skyblue', edgecolor='navy', alpha=0.8)

    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height * 1.01,
                 f'{height:.2f}', ha='center', va='bottom', fontsize=9)

    plt.xlabel("Слой", fontsize=12)
    plt.ylabel("Норма градиента", fontsize=12)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.savefig(f"{save_dir}/grad_hist", dpi=300, bbox_inches='tight')
    plt.close()

def grad_processing(model, tokenizer, sentences1, sentences2, labels, file_name):
    save_dir = f"/content/drive/MyDrive/colab_results//{file_name}"
    os.makedirs(save_dir, exist_ok=True)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

    encodings = tokenizer(sentences1, sentences2, padding=True, truncation=True,
                        max_length=500, return_tensors="pt")
    labels = torch.tensor(labels, dtype=torch.long)

    encodings = encodings.to(device)
    labels = labels.to(device)

    outputs = model(**encodings, labels=labels)
    loss = outputs.loss

    layer_norms = compute_layer_gradient_norms(model, loss)

    norms = list(layer_norms.values())
    grad_var = np.var(norms)
    with open(f"{save_dir}/grad_var.txt", 'w', encoding='utf-8') as f:
        f.write(str(grad_var))
    print(f"Дисперсия: {grad_var:.4f}")

    plot_gradient_norms(layer_norms, file_name)

In [7]:
test_sentence1 = []
for ii in range(15):
    test_sentence1.append(f"{train_dataset[ii]['word']} [SEP] {train_dataset[ii]['sentence1']}")
test_sentence2 = train_dataset[0:15]['sentence2']
test_label = train_dataset[0:15]['label']
print(test_sentence1)
print(test_sentence2)
print(test_label)

['двор [SEP] В нашей деревне осталось от силы двадцать дворов', 'доклад [SEP] Табличка на дверях: «Без доклада не входить»', 'засада [SEP] У нас вообще […] засада с героями, способными дотягивать в жизни до собственного творчества', 'доля [SEP] Он не успел сказать и десятой доли того, что собирался', 'закат [SEP] Теперь, если она не пойдет звонить мужу, успеет до заката к морю', 'демократия [SEP] Европейские демократии', 'заря [SEP] Заря новой эры', 'задержка [SEP] Задержка взгляда на собеседнике', 'дорожка [SEP] Бурые ковровые дорожки заглушали шаги', 'задержка [SEP] Вскоре я почувствовал всю муторность этой процедуры [сбора финансовых документов о тратах] и проклял все на свете, поскольку за несколько дней вынужденной задержки в Катманду, кажется, только этим и занимался', 'дух [SEP] Завертелась в доме веселая коловерть: праздничный стол, праздничный дух, шумные разговоры', 'защита [SEP] Однажды засыпая, погружаясь в замедленный, чуткий сон, Иван Макаров неожиданно понял, что единств

In [8]:
from transformers.activations import ACT2FN
from transformers.cache_utils import Cache
from transformers.modeling_layers import GradientCheckpointingLayer
from transformers.modeling_outputs import BaseModelOutputWithPastAndCrossAttentions
from transformers.processing_utils import Unpack
from transformers.pytorch_utils import apply_chunking_to_forward
from transformers.utils import TransformersKwargs, auto_docstring
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS
from transformers.models.bert.modeling_bert import eager_attention_forward, BertAttention, BertModel, BertSelfAttention, BertEncoder, BertSelfOutput, BertIntermediate, BertOutput, BertLayer, BertPreTrainedModel, BertForSequenceClassification

class BertSelfAttentionPreLayerNorm(nn.Module):
    def __init__(self, config, is_causal=False, layer_idx=None):
        super().__init__()
        if config.hidden_size % config.num_attention_heads != 0 and not hasattr(config, "embedding_size"):
            raise ValueError(
                f"The hidden size ({config.hidden_size}) is not a multiple of the number of attention "
                f"heads ({config.num_attention_heads})"
            )
        self.config = config

        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = int(config.hidden_size / config.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size
        self.scaling = self.attention_head_size**-0.5

        self.query = nn.Linear(config.hidden_size, self.all_head_size)
        self.key = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

        self.is_decoder = config.is_decoder
        self.is_causal = is_causal
        self.layer_idx = layer_idx
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.attention_head_size)
        hidden_states = self.LayerNorm(hidden_states)

        query_layer = self.query(hidden_states).view(*hidden_shape).transpose(1, 2)
        key_layer = self.key(hidden_states).view(*hidden_shape).transpose(1, 2)
        value_layer = self.value(hidden_states).view(*hidden_shape).transpose(1, 2)

        if past_key_values is not None:
            current_past_key_values = past_key_values
            if isinstance(past_key_values, EncoderDecoderCache):
                current_past_key_values = past_key_values.self_attention_cache

            key_layer, value_layer = current_past_key_values.update(key_layer, value_layer, self.layer_idx)

        attention_interface: Callable = ALL_ATTENTION_FUNCTIONS.get_interface(
            self.config._attn_implementation, eager_attention_forward
        )

        attn_output, attn_weights = attention_interface(
            self,
            query_layer,
            key_layer,
            value_layer,
            attention_mask,
            dropout=0.0 if not self.training else self.dropout.p,
            scaling=self.scaling,
            **kwargs,
        )
        attn_output = attn_output.reshape(*input_shape, -1).contiguous()
        return attn_output, attn_weights

class BertSelfOutputPreLayerNorm(BertSelfOutput):
    def __init__(self, config):
        super().__init__(config)

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = hidden_states + input_tensor
        return hidden_states

class BertAttentionPreLayerNorm(BertAttention):
    def __init__(self, config, is_causal=False, layer_idx=None, is_cross_attention=False):
        super().__init__(config, is_causal=is_causal, layer_idx=layer_idx, is_cross_attention=False)
        self.self = BertSelfAttentionPreLayerNorm(config, is_causal=is_causal, layer_idx=layer_idx)
        self.output = BertSelfOutputPreLayerNorm(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        attention_mask = attention_mask if not self.is_cross_attention else encoder_attention_mask
        attention_output, attn_weights = self.self(
            hidden_states,
            encoder_hidden_states=encoder_hidden_states,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            **kwargs,
        )
        attention_output = self.output(attention_output, hidden_states)
        return attention_output, attn_weights

class BertIntermediatePreLayerNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dense = nn.Linear(config.hidden_size, config.intermediate_size)
        if isinstance(config.hidden_act, str):
            self.intermediate_act_fn = ACT2FN[config.hidden_act]
        else:
            self.intermediate_act_fn = config.hidden_act

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        hidden_states = self.LayerNorm(hidden_states)
        hidden_states = self.dense(hidden_states)
        hidden_states = self.intermediate_act_fn(hidden_states)
        return hidden_states


class BertOutputPreLayerNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.intermediate_size, config.hidden_size)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = hidden_states + input_tensor
        return hidden_states


class BertLayerPreLayerNorm(BertLayer):
    def __init__(self, config, layer_idx=None):
        super().__init__(config)
        self.chunk_size_feed_forward = config.chunk_size_feed_forward
        self.seq_len_dim = 1
        self.attention = BertAttentionPreLayerNorm(config, is_causal=config.is_decoder, layer_idx=layer_idx)
        self.intermediate = BertIntermediatePreLayerNorm(config)
        self.output = BertOutputPreLayerNorm(config)

class BertEncoderPreLayerNorm(BertEncoder):
    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self.layer = nn.ModuleList([BertLayerPreLayerNorm(config, layer_idx=i) for i in range(config.num_hidden_layers)])


class BertModelPreLayerNorm(BertModel):
    _no_split_modules = ["BertEmbeddings", "BertLayerPreLayerNorm"]
    _can_record_outputs = {
        "hidden_states": BertLayerPreLayerNorm,
        "attentions": BertSelfAttentionPreLayerNorm,
    }

    def __init__(self, config, add_pooling_layer=True):
        super().__init__(config, add_pooling_layer=add_pooling_layer)
        self.encoder = BertEncoderPreLayerNorm(config)

class BertForSequenceClassificationPreLayerNorm(BertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        self.bert = BertModelPreLayerNorm(config)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        self.post_init()

In [9]:
class BertSelfAttentionHybridNorm(nn.Module):
    def __init__(self, config, is_causal=False, layer_idx=None):
        super().__init__()
        if config.hidden_size % config.num_attention_heads != 0 and not hasattr(config, "embedding_size"):
            raise ValueError(
                f"The hidden size ({config.hidden_size}) is not a multiple of the number of attention "
                f"heads ({config.num_attention_heads})"
            )
        self.config = config

        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = int(config.hidden_size / config.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size
        self.scaling = self.attention_head_size**-0.5

        self.query = nn.Linear(config.hidden_size, self.all_head_size)
        self.key = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

        self.is_decoder = config.is_decoder
        self.is_causal = is_causal
        self.layer_idx = layer_idx
        self.LayerNorm = nn.LayerNorm(self.attention_head_size, eps=config.layer_norm_eps)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.attention_head_size)

        query_layer = self.query(hidden_states).view(*hidden_shape).transpose(1, 2)
        query_layer = self.LayerNorm(query_layer)
        key_layer = self.key(hidden_states).view(*hidden_shape).transpose(1, 2)
        key_layer = self.LayerNorm(key_layer)
        value_layer = self.value(hidden_states).view(*hidden_shape).transpose(1, 2)
        value_layer = self.LayerNorm(value_layer)

        if past_key_values is not None:
            current_past_key_values = past_key_values
            if isinstance(past_key_values, EncoderDecoderCache):
                current_past_key_values = past_key_values.self_attention_cache

            key_layer, value_layer = current_past_key_values.update(key_layer, value_layer, self.layer_idx)

        attention_interface: Callable = ALL_ATTENTION_FUNCTIONS.get_interface(
            self.config._attn_implementation, eager_attention_forward
        )

        attn_output, attn_weights = attention_interface(
            self,
            query_layer,
            key_layer,
            value_layer,
            attention_mask,
            dropout=0.0 if not self.training else self.dropout.p,
            scaling=self.scaling,
            **kwargs,
        )
        attn_output = attn_output.reshape(*input_shape, -1).contiguous()
        return attn_output, attn_weights

class BertAttentionHybridNorm(BertAttention):
    def __init__(self, config, is_causal=False, layer_idx=None, is_cross_attention=False):
        super().__init__(config, is_causal=is_causal, layer_idx=layer_idx, is_cross_attention=False)
        self.self = BertSelfAttentionHybridNorm(config, is_causal=is_causal, layer_idx=layer_idx)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        attention_mask = attention_mask if not self.is_cross_attention else encoder_attention_mask
        attention_output, attn_weights = self.self(
            hidden_states,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            **kwargs,
        )
        attention_output = self.output(attention_output, hidden_states)
        return attention_output, attn_weights

class BertLayerHybridNorm(BertLayer):
    def __init__(self, config, layer_idx=None):
        super().__init__(config)
        self.chunk_size_feed_forward = config.chunk_size_feed_forward
        self.seq_len_dim = 1
        self.attention = BertAttentionHybridNorm(config, is_causal=config.is_decoder, layer_idx=layer_idx)


class BertEncoderHybridNorm(BertEncoder):
    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self.layer = nn.ModuleList([BertLayerHybridNorm(config, layer_idx=i) for i in range(config.num_hidden_layers)])

class BertModelHybridNorm(BertModel):
    _no_split_modules = ["BertEmbeddings", "BertLayerHybridNorm"]
    _can_record_outputs = {
        "hidden_states": BertLayerHybridNorm,
        "attentions": BertSelfAttentionHybridNorm,
    }

    def __init__(self, config, add_pooling_layer=True):
        super().__init__(config, add_pooling_layer=add_pooling_layer)
        self.encoder = BertEncoderHybridNorm(config)

class BertForSequenceClassificationHybridNorm(BertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        self.bert = BertModelHybridNorm(config)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        self.post_init()

In [10]:
class BertSelfAttentionScalingNorm(nn.Module):
    def __init__(self, config, is_causal=False, layer_idx=None):
        super().__init__()
        if config.hidden_size % config.num_attention_heads != 0 and not hasattr(config, "embedding_size"):
            raise ValueError(
                f"The hidden size ({config.hidden_size}) is not a multiple of the number of attention "
                f"heads ({config.num_attention_heads})"
            )
        self.config = config

        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = int(config.hidden_size / config.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size
        self.scaling = self.attention_head_size**-0.5

        self.query = nn.Linear(config.hidden_size, self.all_head_size)
        self.key = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

        self.is_decoder = config.is_decoder
        self.is_causal = is_causal
        self.layer_idx = layer_idx
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

        self.attenuation_coef = 1/((layer_idx + 1) ** 0.5)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.attention_head_size)
        hidden_states = self.LayerNorm(hidden_states) * self.attenuation_coef

        query_layer = self.query(hidden_states).view(*hidden_shape).transpose(1, 2)
        key_layer = self.key(hidden_states).view(*hidden_shape).transpose(1, 2)
        value_layer = self.value(hidden_states).view(*hidden_shape).transpose(1, 2)

        if past_key_values is not None:
            current_past_key_values = past_key_values
            if isinstance(past_key_values, EncoderDecoderCache):
                current_past_key_values = past_key_values.self_attention_cache

            key_layer, value_layer = current_past_key_values.update(key_layer, value_layer, self.layer_idx)

        attention_interface: Callable = ALL_ATTENTION_FUNCTIONS.get_interface(
            self.config._attn_implementation, eager_attention_forward
        )

        attn_output, attn_weights = attention_interface(
            self,
            query_layer,
            key_layer,
            value_layer,
            attention_mask,
            dropout=0.0 if not self.training else self.dropout.p,
            scaling=self.scaling,
            **kwargs,
        )
        attn_output = attn_output.reshape(*input_shape, -1).contiguous()
        return attn_output, attn_weights

class BertSelfOutputScalingNorm(BertSelfOutput):
    def __init__(self, config):
        super().__init__(config)

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = hidden_states + input_tensor
        return hidden_states

class BertAttentionScalingNorm(BertAttention):
    def __init__(self, config, is_causal=False, layer_idx=None, is_cross_attention=False):
        super().__init__(config, is_causal=is_causal, layer_idx=layer_idx, is_cross_attention=False)
        self.self = BertSelfAttentionScalingNorm(config, is_causal=is_causal, layer_idx=layer_idx)
        self.output = BertSelfOutputScalingNorm(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        attention_mask = attention_mask if not self.is_cross_attention else encoder_attention_mask
        attention_output, attn_weights = self.self(
            hidden_states,
            encoder_hidden_states=encoder_hidden_states,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            **kwargs,
        )
        attention_output = self.output(attention_output, hidden_states)
        return attention_output, attn_weights

class BertIntermediateScalingNorm(nn.Module):
    def __init__(self, config, layer_idx):
        super().__init__()
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dense = nn.Linear(config.hidden_size, config.intermediate_size)

        self.attenuation_coef = 1 / ((layer_idx + 1) ** 0.5)

        if isinstance(config.hidden_act, str):
            self.intermediate_act_fn = ACT2FN[config.hidden_act]
        else:
            self.intermediate_act_fn = config.hidden_act

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        hidden_states = self.LayerNorm(hidden_states) * self.attenuation_coef
        hidden_states = self.dense(hidden_states)
        hidden_states = self.intermediate_act_fn(hidden_states)
        return hidden_states


class BertOutputScalingNorm(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.intermediate_size, config.hidden_size)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = hidden_states + input_tensor
        return hidden_states


class BertLayerScalingNorm(BertLayer):
    def __init__(self, config, layer_idx=None):
        super().__init__(config)
        self.chunk_size_feed_forward = config.chunk_size_feed_forward
        self.seq_len_dim = 1
        self.attention = BertAttentionScalingNorm(config, is_causal=config.is_decoder, layer_idx=layer_idx)
        self.intermediate = BertIntermediateScalingNorm(config, layer_idx)
        self.output = BertOutputScalingNorm(config)

class BertEncoderScalingNorm(BertEncoder):
    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self.layer = nn.ModuleList([BertLayerScalingNorm(config, layer_idx=i) for i in range(config.num_hidden_layers)])

class BertModelScalingNorm(BertModel):
    _no_split_modules = ["BertEmbeddings", "BertLayerScalingNorm"]
    _can_record_outputs = {
        "hidden_states": BertLayerScalingNorm,
        "attentions": BertSelfAttentionScalingNorm,
    }

    def __init__(self, config, add_pooling_layer=True):
        super().__init__(config, add_pooling_layer=add_pooling_layer)
        self.encoder = BertEncoderScalingNorm(config)

class BertForSequenceClassificationScalingNorm(BertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        self.bert = BertModelScalingNorm(config)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        self.post_init()

In [11]:
class BertSelfAttentionScalingNorm2(nn.Module):
    def __init__(self, config, is_causal=False, attenuation_coef=None):
        super().__init__()
        if config.hidden_size % config.num_attention_heads != 0 and not hasattr(config, "embedding_size"):
            raise ValueError(
                f"The hidden size ({config.hidden_size}) is not a multiple of the number of attention "
                f"heads ({config.num_attention_heads})"
            )
        self.config = config

        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = int(config.hidden_size / config.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size
        self.scaling = self.attention_head_size**-0.5

        self.query = nn.Linear(config.hidden_size, self.all_head_size)
        self.key = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

        self.is_decoder = config.is_decoder
        self.is_causal = is_causal
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

        self.attenuation_coef = attenuation_coef

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.attention_head_size)
        hidden_states = self.LayerNorm(hidden_states) * self.attenuation_coef

        query_layer = self.query(hidden_states).view(*hidden_shape).transpose(1, 2)
        key_layer = self.key(hidden_states).view(*hidden_shape).transpose(1, 2)
        value_layer = self.value(hidden_states).view(*hidden_shape).transpose(1, 2)

        if past_key_values is not None:
            current_past_key_values = past_key_values
            if isinstance(past_key_values, EncoderDecoderCache):
                current_past_key_values = past_key_values.self_attention_cache

            key_layer, value_layer = current_past_key_values.update(key_layer, value_layer, self.layer_idx)

        attention_interface: Callable = ALL_ATTENTION_FUNCTIONS.get_interface(
            self.config._attn_implementation, eager_attention_forward
        )

        attn_output, attn_weights = attention_interface(
            self,
            query_layer,
            key_layer,
            value_layer,
            attention_mask,
            dropout=0.0 if not self.training else self.dropout.p,
            scaling=self.scaling,
            **kwargs,
        )
        attn_output = attn_output.reshape(*input_shape, -1).contiguous()
        return attn_output, attn_weights

class BertAttentionScalingNorm2(BertAttention):
    def __init__(self, config, is_causal=False, layer_idx=None, is_cross_attention=False, attenuation_coef=None):
        super().__init__(config, is_causal=is_causal, layer_idx=layer_idx, is_cross_attention=False)
        self.self = BertSelfAttentionScalingNorm2(config, is_causal=is_causal, attenuation_coef=attenuation_coef)
        self.output = BertSelfOutputScalingNorm(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        attention_mask = attention_mask if not self.is_cross_attention else encoder_attention_mask
        attention_output, attn_weights = self.self(
            hidden_states,
            encoder_hidden_states=encoder_hidden_states,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            **kwargs,
        )
        attention_output = self.output(attention_output, hidden_states)
        return attention_output, attn_weights

class BertIntermediateScalingNorm2(nn.Module):
    def __init__(self, config, attenuation_coef):
        super().__init__()
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dense = nn.Linear(config.hidden_size, config.intermediate_size)

        self.attenuation_coef = attenuation_coef

        if isinstance(config.hidden_act, str):
            self.intermediate_act_fn = ACT2FN[config.hidden_act]
        else:
            self.intermediate_act_fn = config.hidden_act

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        hidden_states = self.LayerNorm(hidden_states) * self.attenuation_coef
        hidden_states = self.dense(hidden_states)
        hidden_states = self.intermediate_act_fn(hidden_states)
        return hidden_states

class BertLayerScalingNorm2(BertLayer):
    def __init__(self, config, layer_idx=None, num_layers=None, power=0.5):
        super().__init__(config)
        self.chunk_size_feed_forward = config.chunk_size_feed_forward
        self.seq_len_dim = 1
        attenuation_coef = 1 / ((num_layers - layer_idx + 1) ** power)
        self.attention = BertAttentionScalingNorm2(config, is_causal=config.is_decoder, layer_idx=layer_idx, attenuation_coef=attenuation_coef)
        self.intermediate = BertIntermediateScalingNorm2(config, attenuation_coef)
        self.output = BertOutputScalingNorm(config)

class BertEncoderScalingNorm2(BertEncoder):
    def __init__(self, config, power=0.5):
        super().__init__(config)
        self.config = config
        self.layer = nn.ModuleList([BertLayerScalingNorm2(config, i, config.num_hidden_layers, power) for i in range(config.num_hidden_layers)])

class BertModelScalingNorm2(BertModel):
    _no_split_modules = ["BertEmbeddings", "BertLayerScalingNorm2"]
    _can_record_outputs = {
        "hidden_states": BertLayerScalingNorm2,
        "attentions": BertSelfAttentionScalingNorm2,
    }

    def __init__(self, config, add_pooling_layer=True, power=0.5):
        super().__init__(config, add_pooling_layer=add_pooling_layer)
        self.encoder = BertEncoderScalingNorm2(config, power)

class BertForSequenceClassificationScalingNorm2(BertForSequenceClassification):
    def __init__(self, config, power=0.5):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        self.bert = BertModelScalingNorm2(config, power)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        self.post_init()

In [12]:
from transformers.modeling_outputs import SequenceClassifierOutput

class BertForSequenceClassificationScalingNorm2LAYA(BertForSequenceClassificationScalingNorm2):
    def __init__(self, config, power=0.5):
        super().__init__(config, power)
        hidden_size = config.hidden_size

        self.attention_weights = nn.Linear(hidden_size, 1, bias=False)
        self.classifier = nn.Linear(hidden_size, config.num_labels)

    def forward(
            self,
            input_ids: torch.Tensor | None = None,
            attention_mask: torch.Tensor | None = None,
            token_type_ids: torch.Tensor | None = None,
            position_ids: torch.Tensor | None = None,
            inputs_embeds: torch.Tensor | None = None,
            labels: torch.Tensor | None = None,
            **kwargs: Unpack[TransformersKwargs],
        ) -> tuple[torch.Tensor] | SequenceClassifierOutput:
        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            return_dict=True,
            **kwargs,
            output_hidden_states=True,
        )

        hidden_states = outputs.hidden_states[1:]
        stacked_hidden_states = torch.stack(hidden_states, dim=1)
        cls_states = stacked_hidden_states[:, :, 0, :]
        attention_scores = self.attention_weights(cls_states)
        attention_probs = torch.softmax(attention_scores, dim=1)
        aggregated_representation = torch.sum(attention_probs * cls_states, dim=1)
        logits = self.classifier(aggregated_representation)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels.view(-1))

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=None,
            attentions=outputs.attentions,
        )

class BertForSequenceClassificationScalingNorm2LAYAwithHidden(BertForSequenceClassificationScalingNorm2):
    def __init__(self, config, power=0.5):
        super().__init__(config, power)
        hidden_size = config.hidden_size

        self.attention_weights = nn.Linear(hidden_size, 1, bias=False)
        self.classifier = nn.Linear(hidden_size, config.num_labels)

    def forward(
            self,
            input_ids: torch.Tensor | None = None,
            attention_mask: torch.Tensor | None = None,
            token_type_ids: torch.Tensor | None = None,
            position_ids: torch.Tensor | None = None,
            inputs_embeds: torch.Tensor | None = None,
            labels: torch.Tensor | None = None,
            **kwargs: Unpack[TransformersKwargs],
        ) -> tuple[torch.Tensor] | SequenceClassifierOutput:
        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            return_dict=True,
            **kwargs,
            output_hidden_states=True,
        )

        hidden_states = outputs.hidden_states[1:]
        stacked_hidden_states = torch.stack(hidden_states, dim=1)
        cls_states = stacked_hidden_states[:, :, 0, :]
        attention_scores = self.attention_weights(cls_states)
        attention_probs = torch.softmax(attention_scores, dim=1)
        aggregated_representation = torch.sum(attention_probs * cls_states, dim=1)
        logits = self.classifier(aggregated_representation)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels.view(-1))

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

In [13]:
class BertSelfAttentionDeepNorm(BertSelfAttention):
  def __init__(self, config, beta, is_causal=False, layer_idx=None):
        super().__init__(config, is_causal, layer_idx)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)
        self.value.weight.data *= beta

class BertSelfOutputDeepNorm(nn.Module):
    def __init__(self, config, alpha, beta):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dense.weight.data *= beta

        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.alpha = alpha

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + self.alpha * input_tensor)
        return hidden_states

class BertAttentionDeepNorm(BertAttention):
    def __init__(self, config, alpha, beta, is_causal=False, layer_idx=None, is_cross_attention=False):
        super().__init__(config, is_causal, layer_idx, is_cross_attention)
        attention_class = BertSelfAttentionDeepNorm
        self.self = attention_class(config, beta, is_causal=is_causal, layer_idx=layer_idx)
        self.output = BertSelfOutputDeepNorm(config, alpha, beta)

class BertIntermediateDeepNorm(BertIntermediate):
    def __init__(self, config, beta):
        super().__init__(config)
        self.dense = nn.Linear(config.hidden_size, config.intermediate_size)
        self.dense.weight.data *= beta

class BertOutputDeepNorm(BertOutput):
    def __init__(self, config, alpha, beta):
        super().__init__(config)
        self.dense = nn.Linear(config.intermediate_size, config.hidden_size)
        self.dense.weight.data *= beta
        self.alpha = alpha

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + self.alpha * input_tensor)
        return hidden_states

class BertLayerDeepNorm(BertLayer):
    def __init__(self, config, alpha, beta, layer_idx=None):
        super().__init__(config)
        self.attention = BertAttentionDeepNorm(config, alpha, beta, is_causal=config.is_decoder, layer_idx=layer_idx)
        self.intermediate = BertIntermediateDeepNorm(config, beta)
        self.output = BertOutputDeepNorm(config, alpha, beta)

class BertEncoderDeepNorm(BertEncoder):
    def __init__(self, config, alpha, beta):
        super().__init__(config)
        self.config = config
        self.layer = nn.ModuleList([BertLayerDeepNorm(config, alpha, beta, layer_idx=i) for i in range(config.num_hidden_layers)])

class BertModelDeepNorm(BertModel):
    _no_split_modules = ["BertEmbeddings", "BertLayerDeepNorm"]
    _can_record_outputs = {
        "hidden_states": BertLayerDeepNorm,
        "attentions": BertSelfAttentionDeepNorm,
    }

    def __init__(self, config, add_pooling_layer=True):
        super().__init__(config, add_pooling_layer)
        alpha = (2 * config.num_hidden_layers) ** 0.25
        beta = 1 / (8 * config.num_hidden_layers) ** 0.25
        self.encoder = BertEncoderDeepNorm(config, alpha, beta)

class BertForSequenceClassificationDeepNorm(BertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        self.bert = BertModelDeepNorm(config)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        self.post_init()

In [14]:
class BertSelfAttentionPeriNorm(nn.Module):
    def __init__(self, config, is_causal=False, layer_idx=None):
        super().__init__()
        if config.hidden_size % config.num_attention_heads != 0 and not hasattr(config, "embedding_size"):
            raise ValueError(
                f"The hidden size ({config.hidden_size}) is not a multiple of the number of attention "
                f"heads ({config.num_attention_heads})"
            )
        self.config = config

        self.num_attention_heads = config.num_attention_heads
        self.attention_head_size = int(config.hidden_size / config.num_attention_heads)
        self.all_head_size = self.num_attention_heads * self.attention_head_size
        self.scaling = self.attention_head_size**-0.5

        self.query = nn.Linear(config.hidden_size, self.all_head_size)
        self.key = nn.Linear(config.hidden_size, self.all_head_size)
        self.value = nn.Linear(config.hidden_size, self.all_head_size)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

        self.is_decoder = config.is_decoder
        self.is_causal = is_causal
        self.layer_idx = layer_idx
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        input_shape = hidden_states.shape[:-1]
        hidden_shape = (*input_shape, -1, self.attention_head_size)
        hidden_states = self.LayerNorm(hidden_states)

        query_layer = self.query(hidden_states).view(*hidden_shape).transpose(1, 2)
        key_layer = self.key(hidden_states).view(*hidden_shape).transpose(1, 2)
        value_layer = self.value(hidden_states).view(*hidden_shape).transpose(1, 2)

        if past_key_values is not None:
            current_past_key_values = past_key_values
            if isinstance(past_key_values, EncoderDecoderCache):
                current_past_key_values = past_key_values.self_attention_cache

            key_layer, value_layer = current_past_key_values.update(key_layer, value_layer, self.layer_idx)

        attention_interface: Callable = ALL_ATTENTION_FUNCTIONS.get_interface(
            self.config._attn_implementation, eager_attention_forward
        )

        attn_output, attn_weights = attention_interface(
            self,
            query_layer,
            key_layer,
            value_layer,
            attention_mask,
            dropout=0.0 if not self.training else self.dropout.p,
            scaling=self.scaling,
            **kwargs,
        )
        attn_output = attn_output.reshape(*input_shape, -1).contiguous()
        return attn_output, attn_weights

class BertSelfOutputPeriNorm(BertSelfOutput):
    def __init__(self, config):
        super().__init__(config)

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states) + input_tensor
        return hidden_states

class BertAttentionPeriNorm(BertAttention):
    def __init__(self, config, is_causal=False, layer_idx=None, is_cross_attention=False):
        super().__init__(config, is_causal=is_causal, layer_idx=layer_idx, is_cross_attention=False)
        self.self = BertSelfAttentionPeriNorm(config, is_causal=is_causal, layer_idx=layer_idx)
        self.output = BertSelfOutputPeriNorm(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        attention_mask = attention_mask if not self.is_cross_attention else encoder_attention_mask
        attention_output, attn_weights = self.self(
            hidden_states,
            encoder_hidden_states=encoder_hidden_states,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            **kwargs,
        )
        attention_output = self.output(attention_output, hidden_states)
        return attention_output, attn_weights

class BertIntermediatePeriNorm(BertIntermediate):
    def __init__(self, config):
        super().__init__(config)
        self.LayerNorm = nn.LayerNorm(config.hidden_size, eps=config.layer_norm_eps)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        hidden_states = self.LayerNorm(hidden_states)
        hidden_states = self.dense(hidden_states)
        hidden_states = self.intermediate_act_fn(hidden_states)
        return hidden_states


class BertOutputPeriNorm(BertOutput):
    def __init__(self, config):
        super().__init__(config)

    def forward(self, hidden_states: torch.Tensor, input_tensor: torch.Tensor) -> torch.Tensor:
        hidden_states = self.dense(hidden_states)
        hidden_states = self.dropout(hidden_states)
        hidden_states = self.LayerNorm(hidden_states + input_tensor)
        return hidden_states


class BertLayerPeriNorm(BertLayer):
    def __init__(self, config, layer_idx=None):
        super().__init__(config)
        self.chunk_size_feed_forward = config.chunk_size_feed_forward
        self.seq_len_dim = 1
        self.attention = BertAttentionPeriNorm(config, is_causal=config.is_decoder, layer_idx=layer_idx)
        self.intermediate = BertIntermediatePeriNorm(config)
        self.output = BertOutputPeriNorm(config)

class BertEncoderPeriNorm(BertEncoder):
    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self.layer = nn.ModuleList([BertLayerPeriNorm(config, layer_idx=i) for i in range(config.num_hidden_layers)])

class BertModelPeriNorm(BertModel):
    _no_split_modules = ["BertEmbeddings", "BertLayerPeriNorm"]
    _can_record_outputs = {
        "hidden_states": BertLayerPeriNorm,
        "attentions": BertSelfAttentionPeriNorm,
    }

    def __init__(self, config, add_pooling_layer=True):
        super().__init__(config, add_pooling_layer=add_pooling_layer)
        self.encoder = BertEncoderPeriNorm(config)

class BertForSequenceClassificationPeriNorm(BertForSequenceClassification):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        self.bert = BertModelPeriNorm(config)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        self.post_init()

In [15]:
class BertAttentionPeriScalingNorm(BertAttention):
    def __init__(self, config, is_causal=False, layer_idx=None, is_cross_attention=False, attenuation_coef=None):
        super().__init__(config, is_causal=is_causal, layer_idx=layer_idx, is_cross_attention=False)
        self.self = BertSelfAttentionScalingNorm2(config, is_causal=is_causal, attenuation_coef=attenuation_coef)
        self.output = BertSelfOutputPeriNorm(config)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: torch.FloatTensor | None = None,
        encoder_hidden_states: torch.FloatTensor | None = None,
        encoder_attention_mask: torch.FloatTensor | None = None,
        past_key_values: Cache | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor]:
        attention_mask = attention_mask if not self.is_cross_attention else encoder_attention_mask
        attention_output, attn_weights = self.self(
            hidden_states,
            encoder_hidden_states=encoder_hidden_states,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            **kwargs,
        )
        attention_output = self.output(attention_output, hidden_states)
        return attention_output, attn_weights

class BertLayerPeriScalingNorm(BertLayer):
    def __init__(self, config, layer_idx=None, num_layers=None, power=0.5):
        super().__init__(config)
        self.chunk_size_feed_forward = config.chunk_size_feed_forward
        self.seq_len_dim = 1
        attenuation_coef = 1 / ((num_layers - layer_idx + 1) ** power)
        self.attention = BertAttentionPeriScalingNorm(config, is_causal=config.is_decoder, layer_idx=layer_idx, attenuation_coef=attenuation_coef)
        self.intermediate = BertIntermediateScalingNorm2(config, attenuation_coef)
        self.output = BertOutputPeriNorm(config)

class BertEncoderPeriScalingNorm(BertEncoder):
    def __init__(self, config, power=0.5):
        super().__init__(config)
        self.config = config
        self.layer = nn.ModuleList([BertLayerPeriScalingNorm(config, i, config.num_hidden_layers, power) for i in range(config.num_hidden_layers)])

class BertModelPeriScalingNorm(BertModel):
    _no_split_modules = ["BertEmbeddings", "BertLayerPeriScalingNorm"]
    _can_record_outputs = {
        "hidden_states": BertLayerPeriScalingNorm,
        "attentions": BertSelfAttentionScalingNorm2,
    }

    def __init__(self, config, add_pooling_layer=True, power=0.5):
        super().__init__(config, add_pooling_layer=add_pooling_layer)
        self.encoder = BertEncoderPeriScalingNorm(config, power)

class BertForSequenceClassificationPeriScalingNorm(BertForSequenceClassification):
    def __init__(self, config, power=0.5):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        self.bert = BertModelPeriScalingNorm(config, power)
        classifier_dropout = (
            config.classifier_dropout if config.classifier_dropout is not None else config.hidden_dropout_prob
        )
        self.dropout = nn.Dropout(classifier_dropout)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

        self.post_init()

In [16]:
class BertForSequenceClassificationPeriScalingNormLAYA(BertForSequenceClassificationPeriScalingNorm):
    def __init__(self, config, power=0.5):
        super().__init__(config, power)
        hidden_size = config.hidden_size

        self.attention_weights = nn.Linear(hidden_size, 1, bias=False)
        self.classifier = nn.Linear(hidden_size, config.num_labels)

    def forward(
            self,
            input_ids: torch.Tensor | None = None,
            attention_mask: torch.Tensor | None = None,
            token_type_ids: torch.Tensor | None = None,
            position_ids: torch.Tensor | None = None,
            inputs_embeds: torch.Tensor | None = None,
            labels: torch.Tensor | None = None,
            **kwargs: Unpack[TransformersKwargs],
        ) -> tuple[torch.Tensor] | SequenceClassifierOutput:
        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            return_dict=True,
            **kwargs,
            output_hidden_states=True,
        )

        hidden_states = outputs.hidden_states[1:]
        stacked_hidden_states = torch.stack(hidden_states, dim=1)
        cls_states = stacked_hidden_states[:, :, 0, :]
        attention_scores = self.attention_weights(cls_states)
        attention_probs = torch.softmax(attention_scores, dim=1)
        aggregated_representation = torch.sum(attention_probs * cls_states, dim=1)
        logits = self.classifier(aggregated_representation)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels.view(-1))

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=None,
            attentions=outputs.attentions,
        )

class BertForSequenceClassificationPeriScalingNormLAYAwithHidden(BertForSequenceClassificationPeriScalingNorm):
    def __init__(self, config, power=0.5):
        super().__init__(config, power)
        hidden_size = config.hidden_size

        self.attention_weights = nn.Linear(hidden_size, 1, bias=False)
        self.classifier = nn.Linear(hidden_size, config.num_labels)

    def forward(
            self,
            input_ids: torch.Tensor | None = None,
            attention_mask: torch.Tensor | None = None,
            token_type_ids: torch.Tensor | None = None,
            position_ids: torch.Tensor | None = None,
            inputs_embeds: torch.Tensor | None = None,
            labels: torch.Tensor | None = None,
            **kwargs: Unpack[TransformersKwargs],
        ) -> tuple[torch.Tensor] | SequenceClassifierOutput:
        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            inputs_embeds=inputs_embeds,
            return_dict=True,
            **kwargs,
            output_hidden_states=True,
        )

        hidden_states = outputs.hidden_states[1:]
        stacked_hidden_states = torch.stack(hidden_states, dim=1)
        cls_states = stacked_hidden_states[:, :, 0, :]
        attention_scores = self.attention_weights(cls_states)
        attention_probs = torch.softmax(attention_scores, dim=1)
        aggregated_representation = torch.sum(attention_probs * cls_states, dim=1)
        logits = self.classifier(aggregated_representation)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels.view(-1))

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

In [17]:
from torchmetrics.classification import F1Score

def find_f1(model, tokenizer, dataloader):
    f1 = F1Score(task="binary").to(device)
    model.eval()
    with torch.no_grad():
        for batch in dataloader:
            combined = []
            for ii in range(len(batch['word'])):
                combined.append(f"{batch['word'][ii]} [SEP] {batch['sentence1'][ii]}")
            inputs = tokenizer(combined, batch['sentence2'], return_tensors="pt", padding=True)
            inputs = inputs.to(device)
            labels = batch['label']
            labels = labels.to(device)

            outputs = model(**inputs)
            preds = torch.tensor([0 if o[0] > o[1] else 1 for o in outputs.logits])
            preds = preds.to(device)
            f1(preds, labels)

    common_f1 = f1.compute()
    return common_f1

def train_model_early_stop(model, tokenizer, model_name, train_dataloader, val_dataloader, optimizer, loss_fn, save_model = 0, max_epochs = 100, patience = 3):
    best_val_f1 = -1
    epochs_no_improve = 0
    loss_epochs = []
    for epoch in range(max_epochs):
        f1 = F1Score(task="binary").to(device)
        loss_points = np.array([])
        for batch in train_dataloader:
            model.train()
            combined = []
            for ii in range(len(batch['word'])):
                combined.append(f"{batch['word'][ii]} [SEP] {batch['sentence1'][ii]}")
            inputs = tokenizer(combined, batch['sentence2'], return_tensors="pt", padding=True)
            inputs = inputs.to(device)
            labels = batch['label']
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(**inputs)
            logits = outputs.logits
            loss = loss_fn(logits, labels)
            print(loss)
            loss_points = np.append(loss_points, loss.item())

            preds = torch.tensor([0 if o[0] > o[1] else 1 for o in logits])
            preds = preds.to(device)
            f1(preds, labels)
            loss.backward()
            optimizer.step()

        loss_epochs.append(np.mean(loss_points))
        test_f1 = f1.compute()
        val_f1 = find_f1(model, tokenizer, val_dataloader)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            if save_model:
                save_dir = f"/content/drive/MyDrive/colab_results/models/{model_name}"
                os.makedirs(save_dir, exist_ok=True)
                model.save_pretrained(save_dir)
        else:
            epochs_no_improve += 1

        if epochs_no_improve > patience:
            print("Early stopping")
            break

        print(f"{epoch} epoch: train {str(test_f1)}, validation {str(val_f1)}")

        save_dir = f"/content/drive/MyDrive/colab_results/{model_name}"
        os.makedirs(save_dir, exist_ok=True)
        all_layer_states = get_all_layer_hidden_states(model, test_sentence1, test_sentence2, tokenizer)
        #ang_matr_processing(all_layer_states, f"{model_name}/{epoch + 1}")
        ang_hist_processing(all_layer_states, f"{model_name}/{epoch + 1}")
        grad_processing(model, tokenizer, test_sentence1, test_sentence2, test_label, f"{model_name}/{epoch + 1}")

        plt.plot([ii for ii in range(len(loss_epochs))], loss_epochs)
        plt.grid()
        plt.tight_layout()
        plt.savefig(f'/content/drive/MyDrive/colab_results/{model_name}/loss_graph', dpi=300, bbox_inches='tight')
        plt.close()

        with open(f'/content/drive/MyDrive/colab_results/{model_name}/loss_array.txt', 'w', encoding='utf-8') as f:
             f.write(str(loss_epochs))
        with open(f'/content/drive/MyDrive/colab_results/{model_name}/F1_{epoch + 1}.txt', 'w', encoding='utf-8') as f:
             f.write(f"train: {str(test_f1)}, validation: {str(val_f1)}")

In [18]:
def build_optimizer_with_layer_lrs(model, base_lr=1e-5, laya_lr=1e-3, weight_decay=1e-2):
    laya_param_names = {"attention_weights", "classifier"}

    laya_params, inner_params = [], []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        top = name.split(".")[0]
        if top in laya_param_names:
            laya_params.append(param)
        else:
            inner_params.append(param)

    optimizer = torch.optim.AdamW(
        [
            {"params": inner_params, "lr": base_lr, "weight_decay": weight_decay},
            {"params": laya_params, "lr": laya_lr, "weight_decay": 0.0},
        ]
    )
    return optimizer

In [ ]:
config = BertConfig(
    vocab_size=130000,
    num_hidden_layers=32,
    hidden_size=128,
    num_attention_heads=8,
    max_position_embeddings=1024,
    num_labels=2,
)
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")

SEED = 42

set_seed(SEED)
model = BertForSequenceClassificationScalingNorm2LAYA(config).to(device)
optimizer = optimizer = build_optimizer_with_layer_lrs(model)

train_model_early_stop(model, tokenizer, "russe/32layers_config/patience3/seed42/LAYAbert(power=0.5, base_lr=1e-5, laya_lr=1e-3)", train_dataloader, val_dataloader, optimizer, loss_fn, 1)

set_seed(SEED)
model = BertForSequenceClassificationScalingNorm2LAYA(config).to(device)
optimizer = optimizer = build_optimizer_with_layer_lrs(model, base_lr=2e-5)

train_model_early_stop(model, tokenizer, "russe/32layers_config/patience3/seed42/LAYAbert(power=0.5, base_lr=2e-5, laya_lr=1e-3)", train_dataloader, val_dataloader, optimizer, loss_fn, 1)

set_seed(SEED)
model = BertForSequenceClassificationScalingNorm2LAYA(config, power=0.6).to(device)
optimizer = optimizer = build_optimizer_with_layer_lrs(model)

train_model_early_stop(model, tokenizer, "russe/32layers_config/patience3/seed42/LAYAbert(power=0.6, base_lr=1e-5, laya_lr=1e-3)", train_dataloader, val_dataloader, optimizer, loss_fn, 1)

set_seed(SEED)
model = BertForSequenceClassificationScalingNorm2LAYA(config).to(device)
optimizer = optimizer = build_optimizer_with_layer_lrs(model, laya_lr=1e-4)

train_model_early_stop(model, tokenizer, "russe/32layers_config/patience3/seed42/LAYAbert(power=0.5, base_lr=1e-5, laya_lr=1e-4)", train_dataloader, val_dataloader, optimizer, loss_fn, 1)

SEED = 84

set_seed(SEED)
model = BertForSequenceClassificationScalingNorm2LAYA(config).to(device)
optimizer = optimizer = build_optimizer_with_layer_lrs(model)

train_model_early_stop(model, tokenizer, "russe/32layers_config/patience3/seed84/LAYAbert(power=0.5, base_lr=1e-5, laya_lr=1e-3)", train_dataloader, val_dataloader, optimizer, loss_fn, 1)

set_seed(SEED)
model = BertForSequenceClassificationScalingNorm2LAYA(config).to(device)
optimizer = optimizer = build_optimizer_with_layer_lrs(model, base_lr=2e-5)

train_model_early_stop(model, tokenizer, "russe/32layers_config/patience3/seed84/LAYAbert(power=0.5, base_lr=2e-5, laya_lr=1e-3)", train_dataloader, val_dataloader, optimizer, loss_fn, 1)

set_seed(SEED)
model = BertForSequenceClassificationScalingNorm2LAYA(config, power=0.6).to(device)
optimizer = optimizer = build_optimizer_with_layer_lrs(model)

train_model_early_stop(model, tokenizer, "russe/32layers_config/patience3/seed84/LAYAbert(power=0.6, base_lr=1e-5, laya_lr=1e-3)", train_dataloader, val_dataloader, optimizer, loss_fn, 1)

set_seed(SEED)
model = BertForSequenceClassificationScalingNorm2LAYA(config).to(device)
optimizer = optimizer = build_optimizer_with_layer_lrs(model, laya_lr=1e-4)

train_model_early_stop(model, tokenizer, "russe/32layers_config/patience3/seed84/LAYAbert(power=0.5, base_lr=1e-5, laya_lr=1e-4)", train_dataloader, val_dataloader, optimizer, loss_fn, 1)


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

tensor(0.7575, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.7562, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6852, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6647, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6907, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.7268, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.7161, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.7077, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6557, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.7019, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6235, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.7765, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.7406, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6837, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5995, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6980, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6293, device='cuda:0', grad_fn=

...

13 epoch: train tensor(0.6986, device='cuda:0'), validation tensor(0.5970, device='cuda:0')
Дисперсия норм выходных активаций: 601.0636
Дисперсия: 0.1133
tensor(0.4408, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.2984, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3418, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4656, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5040, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4062, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4361, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5092, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5176, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3201, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3643, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5930, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4417, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3938, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

14 epoch: train tensor(0.7134, device='cuda:0'), validation tensor(0.6015, device='cuda:0')
Дисперсия норм выходных активаций: 726.5050
Дисперсия: 0.0484
tensor(0.4733, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.2786, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3661, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4477, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3696, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5331, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4390, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4612, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4832, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3039, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4274, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5162, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3318, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3429, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

15 epoch: train tensor(0.7155, device='cuda:0'), validation tensor(0.6049, device='cuda:0')
Дисперсия норм выходных активаций: 798.3932
Дисперсия: 0.1433
tensor(0.3874, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3712, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4146, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4116, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4253, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3920, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4484, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5402, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3378, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4009, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4237, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3780, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3974, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5991, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor

In [ ]:
config = BertConfig(
    vocab_size=130000,
    num_hidden_layers=32,
    hidden_size=128,
    num_attention_heads=8,
    max_position_embeddings=1024,
    num_labels=2,
)
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")
SEED = 96

set_seed(SEED)
model = BertForSequenceClassificationScalingNorm(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/32layers_config/patience3/seed96/scalingbert", train_dataloader, val_dataloader, optimizer, loss_fn)

set_seed(SEED)
model = BertForSequenceClassificationScalingNorm2(config, power=0.4).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/32layers_config/patience3/seed96/scalingbert(modif, power=0.4)", train_dataloader, val_dataloader, optimizer, loss_fn)

SEED = 24

set_seed(SEED)
model = BertForSequenceClassification(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/32layers_config/patience3/seed24/bert", train_dataloader, val_dataloader, optimizer, loss_fn)

set_seed(SEED)
model = BertForSequenceClassificationScalingNorm(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/32layers_config/patience3/seed24/scalingbert", train_dataloader, val_dataloader, optimizer, loss_fn)

set_seed(SEED)
model = BertForSequenceClassificationScalingNorm2(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/32layers_config/patience3/seed24/scalingbert(modif)", train_dataloader, val_dataloader, optimizer, loss_fn)

set_seed(SEED)
model = BertForSequenceClassificationScalingNorm2(config, power=0.75).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/32layers_config/patience3/seed24/scalingbert(modif, power=0.75)", train_dataloader, val_dataloader, optimizer, loss_fn)

set_seed(SEED)
model = BertForSequenceClassificationPreLayerNorm(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/32layers_config/patience3/seed24/prebert", train_dataloader, val_dataloader, optimizer, loss_fn)


In [ ]:
config = BertConfig(
    vocab_size=130000,
    num_hidden_layers=32,
    hidden_size=128,
    num_attention_heads=8,
    max_position_embeddings=1024,
    num_labels=2,
)
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")
SEED = 84

set_seed(SEED)
model = BertForSequenceClassificationPeriScalingNorm(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/32layers_config/seed84/peri_scalingbert(fixed)", train_dataloader, val_dataloader, optimizer, loss_fn)

SEED = 42

set_seed(SEED)
model = BertForSequenceClassificationPeriScalingNorm(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/32layers_config/seed42/peri_scalingbert(fixed)", train_dataloader, val_dataloader, optimizer, loss_fn)

SEED = 24

set_seed(SEED)
model = BertForSequenceClassificationPeriScalingNorm(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/32layers_config/seed24/peri_scalingbert(fixed)", train_dataloader, val_dataloader, optimizer, loss_fn)

SEED = 96

set_seed(SEED)
model = BertForSequenceClassificationPeriScalingNorm(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "russe/32layers_config/seed96/peri_scalingbert(fixed)", train_dataloader, val_dataloader, optimizer, loss_fn)


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Выходные данные были обрезаны до нескольких последних строк (5000).
tensor(0.4377, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4946, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4406, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3986, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4917, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5845, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4562, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5070, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6096, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4074, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5995, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4501, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4445, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5677, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4694, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5111, device='cuda:0', 

In [16]:
config = BertConfig(
    vocab_size=130000,
    num_hidden_layers=32,
    hidden_size=128,
    num_attention_heads=8,
    max_position_embeddings=1024,
    num_labels=2,
)
SEED=42
tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")
set_seed(SEED)
model = BertForSequenceClassificationPreLayerNorm(config).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

train_model_early_stop(model, tokenizer, "test/prebert(new)", train_dataloader, val_dataloader, optimizer, loss_fn, max_epochs = 10)

tensor(0.7094, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6917, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6808, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6784, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.7074, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6584, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6788, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6552, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6784, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6679, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6979, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6926, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6927, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5786, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6509, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5887, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6743, device='cuda:0', grad_fn=

In [ ]:
from google.colab import runtime
runtime.unassign()